In [1]:
# Télécharge, via l'API transport.data.gouv.fr (même routine que
# scripts/rafraichir_gtfs.py : recuperer_datasets_public_transit +
# resultat_pour_page_url, pas un simple re-téléchargement de
# ressource_url en cache — resultat_pour_page_url ré-interroge le PAN
# pour retrouver la ressource *actuelle* du dataset, potentiellement
# différente si la ressource a changé d'URL depuis le dernier
# enregistrement de provenance), tous les GTFS déjà associés à un
# dataset dans data/gtfs_sources.json — dans data/GTFS_temp/, PAS
# data/GTFS/ (ne remplace rien automatiquement, à l'écart pour
# inspection/comparaison manuelle avant de basculer si besoin).
#
# Les GTFS de gtfs_sources.json sans page_url (associés seulement pour
# leur académie/zone, cf. src.vacances_scolaires — jamais liés à un
# dataset PAN) sont ignorés : rien à télécharger pour eux ici.

import os

from src.transport_data_gouv import (
    charger_provenance,
    recuperer_datasets_public_transit,
    resultat_pour_page_url,
    telecharger_gtfs,
)

DOSSIER_TEMP = os.path.join("data", "GTFS_temp")
os.makedirs(DOSSIER_TEMP, exist_ok=True)

provenance = charger_provenance()
a_telecharger = {f: info for f, info in provenance.items() if info.get("page_url")}
print(f"{len(a_telecharger)} GTFS liés à transport.data.gouv.fr à télécharger dans {DOSSIER_TEMP}\n")

print("Récupération du catalogue transport.data.gouv.fr...")
datasets = recuperer_datasets_public_transit()

reussis, introuvables, echecs = [], [], []
for nom_fichier, info in sorted(a_telecharger.items()):
    resultat = resultat_pour_page_url(info["page_url"], info.get("ressource_url"), datasets)
    if resultat is None:
        print(f"⚠ {nom_fichier} : dataset introuvable sur transport.data.gouv.fr (page supprimée/déplacée ?)")
        introuvables.append(nom_fichier)
        continue
    try:
        contenu = telecharger_gtfs(resultat)
        chemin_cible = os.path.join(DOSSIER_TEMP, nom_fichier)
        with open(chemin_cible, "wb") as f:
            f.write(contenu)
        print(f"✓ {nom_fichier} ({len(contenu) / 1e6:.1f} Mo, màj {resultat['ressource_maj']})")
        reussis.append(nom_fichier)
    except Exception as e:
        print(f"✗ {nom_fichier} : {type(e).__name__}: {e}")
        echecs.append(nom_fichier)

print(f"\n{len(reussis)} réussi(s), {len(introuvables)} introuvable(s), {len(echecs)} échec(s)")
if introuvables:
    print("Introuvables :", introuvables)
if echecs:
    print("Échecs :", echecs)

57 GTFS liés à transport.data.gouv.fr à télécharger dans data/GTFS_temp

Récupération du catalogue transport.data.gouv.fr...
✓ Albi_GTFS.zip (0.0 Mo, màj 2026-06-22T08:47:55.785000Z)
✓ Alès_GTFS.zip (0.3 Mo, màj 2026-08-20T07:41:09.194000Z)
✓ Amiens_GTFS.zip (3.0 Mo, màj 2026-07-22T09:10:44.162000Z)
✓ Angers_GTFS.zip (4.1 Mo, màj 2026-08-15T00:01:10.000000Z)
✓ Avignon_GTFS.zip (2.8 Mo, màj 2026-08-21T09:57:50.000000Z)
✓ Bayonne_GTFS.zip (5.5 Mo, màj 2026-08-29T22:01:19.162610Z)
✓ Besançon_GTFS.zip (4.2 Mo, màj 2026-08-01T01:05:01.000000Z)
✓ Bordeaux_GTFS.zip (19.0 Mo, màj 2026-08-29T22:53:35.914224Z)
✓ Bourges_GTFS.zip (1.7 Mo, màj 2026-07-27T18:50:37.000000Z)
✓ Brest_GTFS.zip (2.6 Mo, màj 2026-08-27T09:21:21.000000Z)
✓ Castres_GTFS.zip (0.5 Mo, màj 2026-01-08T09:53:09.959000Z)
✓ Chambéry_GTFS.zip (2.1 Mo, màj 2026-08-23T16:47:03.074942Z)
✓ Cherbourg-en-Cotentin_GTFS.zip (2.0 Mo, màj 2026-07-01T12:30:51.571000Z)
✓ Cholet_GTFS.zip (1.2 Mo, màj 2026-07-28T15:37:40.317000Z)
✓ Châteauroux_

In [2]:
# Calcule, pour chaque GTFS téléchargé dans data/GTFS_temp/ (cellule
# précédente), sa période de validité (date_debut/date_fin) et sa date
# JOB (dernier mardi/jeudi hors vacances scolaires de l'académie si
# connue, cf. src.info_reseau.dates_service) — écrit dans
# data/GTFS_temp/gtfs_sources_temp.json pour comparer avant de basculer
# ces fichiers vers data/GTFS/ (rien n'est remplacé automatiquement ici).

import json

from src.info_reseau import dates_service
from src.utils import charger_gtfs
from src.vacances_scolaires import departement_academie_zone_pour_feed

fichiers = sorted(f for f in os.listdir(DOSSIER_TEMP) if f.lower().endswith(".zip"))
print(f"{len(fichiers)} GTFS à analyser dans {DOSSIER_TEMP}\n")

resultats = {}
for nom_fichier in fichiers:
    chemin = os.path.join(DOSSIER_TEMP, nom_fichier)
    try:
        feed = charger_gtfs(chemin)
    except Exception as e:
        print(f"✗ {nom_fichier} : impossible de charger ({type(e).__name__}: {e})")
        continue

    try:
        _, academie, _ = departement_academie_zone_pour_feed(feed)
    except Exception:
        academie = None

    try:
        _, date_debut, date_fin, date_job = dates_service(feed, academie=academie)
    except Exception as e:
        print(f"✗ {nom_fichier} : échec dates_service ({type(e).__name__}: {e})")
        continue

    resultats[nom_fichier] = {
        "academie": academie,
        "date_debut": date_debut,
        "date_fin": date_fin,
        "date_JOB": date_job,
    }
    print(f"✓ {nom_fichier} : {date_debut} -> {date_fin} (JOB {date_job})")

chemin_json = os.path.join(DOSSIER_TEMP, "gtfs_sources_temp.json")
with open(chemin_json, "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2, sort_keys=True)

print(f"\n{len(resultats)}/{len(fichiers)} GTFS analysé(s) — écrit dans {chemin_json}")


55 GTFS à analyser dans data/GTFS_temp

Chargement du fichier GTFS : data/GTFS_temp/Albi_GTFS.zip
✓ GTFS chargé avec succès
✓ Albi_GTFS.zip : 20260601 -> 20261231 (JOB 20261217)
Chargement du fichier GTFS : data/GTFS_temp/Alès_GTFS.zip
✓ GTFS chargé avec succès
✓ Alès_GTFS.zip : 20260901 -> 20261016 (JOB 20261015)
Chargement du fichier GTFS : data/GTFS_temp/Amiens_GTFS.zip
✓ GTFS chargé avec succès
✓ Amiens_GTFS.zip : 20260715 -> 20270112 (JOB 20270112)
Chargement du fichier GTFS : data/GTFS_temp/Angers_GTFS.zip
✓ GTFS chargé avec succès
✓ Angers_GTFS.zip : 20260814 -> 20261031 (JOB 20261015)
Chargement du fichier GTFS : data/GTFS_temp/Avignon_GTFS.zip
✓ GTFS chargé avec succès
✓ Avignon_GTFS.zip : 20260824 -> 20261017 (JOB 20261015)
Chargement du fichier GTFS : data/GTFS_temp/Bayonne_GTFS.zip
✓ GTFS chargé avec succès
✓ Bayonne_GTFS.zip : 20260826 -> 20270625 (JOB 20270624)
Chargement du fichier GTFS : data/GTFS_temp/Besançon_GTFS.zip
✓ GTFS chargé avec succès
✓ Besançon_GTFS.zip : 20

In [3]:
# compare les fichiers data/GTFS_temp/gtfs_sources_temp.json et
# data/gtfs_sources.json pour identifier les GTFS obsolètes dans data/GTFS
# (date_JOB actuelle inférieure à date_JOB du GTFS téléchargé).
#
# data/gtfs_sources.json ne stocke pas date_JOB (seulement la provenance
# PAN — page_url/ressource_url/titre) : sert ici de référence pour le
# titre du jeu de données de chaque fichier dans le résultat, la date_JOB
# "actuelle" est recalculée sur le GTFS effectivement présent dans
# data/GTFS/ (même routine academie-aware que la cellule précédente) —
# seule source fiable de ce que produirait l'app avec le fichier actuel.
#
# Résultat dans data/GTFS_temp/compare_GTFS.csv : format tabulaire, le
# plus adapté pour comparer/filtrer ces résultats (ex. dans un tableur),
# contrairement au JSON des deux fichiers comparés ci-dessus.

import csv

GTFS_DIR = os.path.join("data", "GTFS")

with open(os.path.join(DOSSIER_TEMP, "gtfs_sources_temp.json"), encoding="utf-8") as f:
    resultats_temp = json.load(f)

provenance = charger_provenance()

lignes_comparaison = []
for nom_fichier, info_temp in sorted(resultats_temp.items()):
    chemin_local = os.path.join(GTFS_DIR, nom_fichier)
    if not os.path.exists(chemin_local):
        print(f"⚠ {nom_fichier} : absent de data/GTFS/ — pas de comparaison possible")
        continue

    try:
        feed_local = charger_gtfs(chemin_local)
    except Exception as e:
        print(f"✗ {nom_fichier} : impossible de charger la version locale ({type(e).__name__}: {e})")
        continue

    try:
        _, academie_local, _ = departement_academie_zone_pour_feed(feed_local)
    except Exception:
        academie_local = None

    try:
        _, date_debut_local, date_fin_local, date_job_local = dates_service(feed_local, academie=academie_local)
    except Exception as e:
        print(f"✗ {nom_fichier} : échec dates_service sur la version locale ({type(e).__name__}: {e})")
        continue

    date_job_temp = info_temp["date_JOB"]
    obsolete = date_job_local < date_job_temp

    lignes_comparaison.append({
        "nom_fichier": nom_fichier,
        "titre": provenance.get(nom_fichier, {}).get("titre", ""),
        "date_JOB_actuel": date_job_local,
        "date_JOB_temp": date_job_temp,
        "obsolete": obsolete,
    })
    marqueur = "⚠ OBSOLÈTE" if obsolete else "✓ à jour"
    print(f"{marqueur} {nom_fichier} : actuel {date_job_local} vs temp {date_job_temp}")

chemin_csv = os.path.join(DOSSIER_TEMP, "compare_GTFS.csv")
with open(chemin_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["nom_fichier", "titre", "date_JOB_actuel", "date_JOB_temp", "obsolete"])
    writer.writeheader()
    writer.writerows(lignes_comparaison)

nb_obsoletes = sum(1 for l in lignes_comparaison if l["obsolete"])
print(f"\n{len(lignes_comparaison)} comparé(s), {nb_obsoletes} obsolète(s) — écrit dans {chemin_csv}")


Chargement du fichier GTFS : data/GTFS/Albi_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Albi_GTFS.zip : actuel 20261217 vs temp 20261217
Chargement du fichier GTFS : data/GTFS/Alès_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Alès_GTFS.zip : actuel 20261015 vs temp 20261015
Chargement du fichier GTFS : data/GTFS/Amiens_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Amiens_GTFS.zip : actuel 20270112 vs temp 20270112
Chargement du fichier GTFS : data/GTFS/Angers_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Angers_GTFS.zip : actuel 20261015 vs temp 20261015
Chargement du fichier GTFS : data/GTFS/Avignon_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Avignon_GTFS.zip : actuel 20261015 vs temp 20261015
Chargement du fichier GTFS : data/GTFS/Bayonne_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Bayonne_GTFS.zip : actuel 20270624 vs temp 20270624
Chargement du fichier GTFS : data/GTFS/Besançon_GTFS.zip
✓ GTFS chargé avec succès
✓ à jour Besançon_GTFS.zip : actuel 20261015 vs temp 20261015
Chargement du

In [4]:
# écrase les GTFS obsolètes (data/GTFS_temp/compare_GTFS.csv, obsolete=True)
# par leur version fraîchement téléchargée dans data/GTFS_temp/ — écrit
# directement dans data/GTFS/, pousse sur le dataset HF, et invalide les
# caches dérivés (découpage communal, carroyage, extrait OSM, matrice des
# temps de trajet) pour que l'app ne serve pas un résultat périmé — même
# routine que scripts/rafraichir_gtfs.py. ATTENTION : écrase les fichiers
# "en production" (data/GTFS/) et pousse sur HF, relis compare_GTFS.csv
# avant d'exécuter cette cellule. Le recalcul des indicateurs
# d'accessibilité lui-même n'a PAS lieu ici (coûteux : r5py, Overpass) —
# au prochain passage dans l'app ou le notebook, avec le GTFS à jour.

import csv
import shutil

from huggingface_hub import HfApi

from src.hf_cache import HF_DATA_REPO_ID, envoyer_vers_hf
from src.info_reseau import nom_reseau_str as calculer_nom_reseau_str

# Doit rester synchronisé avec GTFS_NOM_RESEAU_FORCE dans app.py et
# NOMS_RESEAU_FORCES dans scripts/rafraichir_gtfs.py.
NOMS_RESEAU_FORCES = {
    "IDFM-gtfs_metro-rer-bus-tram_paris-petite-couronne.zip": "IDFM",
    "IDFM-gtfs.zip": "IDFM",
    "Aix_Marseille_mamp_GTFS.zip": "Aix_Marseille",
}

CHEMINS_CACHE_HF_A_INVALIDER = [
    "memory_csv_agglo/decoupage_agglo_%s.csv",
    "memory_gpkg/population_grid_agglo_%s.gpkg",
    "memory_pbf/agglo_osm_pbf_%s.osm.pbf",
    "memory_ttm/ttm_%s.parquet",
]


def _invalider_caches_derives(nom_reseau):
    api = HfApi()
    for gabarit in CHEMINS_CACHE_HF_A_INVALIDER:
        chemin_hf = gabarit % nom_reseau

        # Copie LOCALE d'abord : recuperer_depuis_hf() est un no-op si le
        # fichier local existe déjà (cf. src/hf_cache.py), donc une
        # invalidation HF seule ne suffit pas — un ttm local encore dans la
        # fenêtre de fraîcheur de 10 jours (cf. cellule "Retourne différentes
        # dates" du notebook principal, ttm_cache_recent) serait rechargé tel
        # quel au prochain run sans jamais voir que le GTFS a changé.
        chemin_local = os.path.join("data", chemin_hf)
        if os.path.exists(chemin_local):
            os.remove(chemin_local)
            print(f"    ✓ cache local supprimé : {chemin_local}")

        try:
            api.delete_file(path_in_repo=chemin_hf, repo_id=HF_DATA_REPO_ID, repo_type="dataset", token=os.environ.get("HF_TOKEN"))
            print(f"    ✓ cache HF invalidé : {chemin_hf}")
        except Exception as e:
            print(f"    (rien à invalider sur HF pour {chemin_hf} : {type(e).__name__})")


with open(os.path.join(DOSSIER_TEMP, "compare_GTFS.csv"), encoding="utf-8") as f:
    lignes_comparaison = list(csv.DictReader(f))

obsoletes = [l["nom_fichier"] for l in lignes_comparaison if l["obsolete"] == "True"]
print(f"{len(obsoletes)} GTFS obsolète(s) à écraser : {obsoletes}\n")

for nom_fichier in obsoletes:
    chemin_temp = os.path.join(DOSSIER_TEMP, nom_fichier)
    chemin_local = os.path.join(GTFS_DIR, nom_fichier)
    shutil.copy(chemin_temp, chemin_local)
    envoyer_vers_hf(chemin_local, f"GTFS/{nom_fichier}")
    print(f"✓ {nom_fichier} écrasé localement + poussé sur HF")

    try:
        feed_maj = charger_gtfs(chemin_local)
        nom_reseau = NOMS_RESEAU_FORCES.get(nom_fichier) or str(calculer_nom_reseau_str(feed_maj))
        print(f"  Invalidation des caches dérivés pour '{nom_reseau}'...")
        _invalider_caches_derives(nom_reseau)
    except Exception as e:
        print(f"  ⚠ impossible de déterminer le réseau pour invalider les caches ({type(e).__name__}: {e})")

print(f"\n{len(obsoletes)} GTFS mis à jour. Relance l'analyse (app ou scripts/run_benchmark_batch.py) pour recalculer leurs indicateurs.")


10 GTFS obsolète(s) à écraser : ['Bordeaux_GTFS.zip', 'Clermont-Ferrand_GTFS.zip', 'Grenoble_GTFS.zip', 'Laval_GTFS.zip', 'Metz_GTFS.zip', 'Nantes_GTFS.zip', 'Nice_GTFS.zip', 'Perpignan_GTFS.zip', 'Strasbourg_GTFS.zip', 'Toulouse_GTFS.zip']



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing Files (1 / 1): 100%|██████████| 19.0MB / 19.0MB, 1.63MB/s  
New Data Upload: 100%|██████████| 19.0MB / 19.0MB, 1.63MB/s  


✓ Bordeaux_GTFS.zip écrasé localement + poussé sur HF
Chargement du fichier GTFS : data/GTFS/Bordeaux_GTFS.zip
✓ GTFS chargé avec succès
  Invalidation des caches dérivés pour 'TBM'...
    ✓ cache invalidé : memory_csv_agglo/decoupage_agglo_TBM.csv
    ✓ cache invalidé : memory_gpkg/population_grid_agglo_TBM.gpkg
    ✓ cache invalidé : memory_pbf/agglo_osm_pbf_TBM.osm.pbf
    ✓ cache invalidé : memory_ttm/ttm_TBM.parquet


Processing Files (1 / 1): 100%|██████████| 7.93MB / 7.93MB,  767kB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


✓ Clermont-Ferrand_GTFS.zip écrasé localement + poussé sur HF
Chargement du fichier GTFS : data/GTFS/Clermont-Ferrand_GTFS.zip
✓ GTFS chargé avec succès
  Invalidation des caches dérivés pour 'T2C'...
    ✓ cache invalidé : memory_csv_agglo/decoupage_agglo_T2C.csv
    ✓ cache invalidé : memory_gpkg/population_grid_agglo_T2C.gpkg
    ✓ cache invalidé : memory_pbf/agglo_osm_pbf_T2C.osm.pbf
    ✓ cache invalidé : memory_ttm/ttm_T2C.parquet


...

✓ Perpignan_GTFS.zip écrasé localement + poussé sur HF
Chargement du fichier GTFS : data/GTFS/Perpignan_GTFS.zip
✓ GTFS chargé avec succès
  Invalidation des caches dérivés pour 'Sankéo'...
    ✓ cache invalidé : memory_csv_agglo/decoupage_agglo_Sankéo.csv
    ✓ cache invalidé : memory_gpkg/population_grid_agglo_Sankéo.gpkg
    ✓ cache invalidé : memory_pbf/agglo_osm_pbf_Sankéo.osm.pbf
    ✓ cache invalidé : memory_ttm/ttm_Sankéo.parquet


Processing Files (1 / 1): 100%|██████████| 5.46MB / 5.46MB,  493kB/s  
New Data Upload: 100%|██████████| 5.46MB / 5.46MB,  493kB/s  


✓ Strasbourg_GTFS.zip écrasé localement + poussé sur HF
Chargement du fichier GTFS : data/GTFS/Strasbourg_GTFS.zip
✓ GTFS chargé avec succès
  Invalidation des caches dérivés pour 'CTS'...
    ✓ cache invalidé : memory_csv_agglo/decoupage_agglo_CTS.csv
    ✓ cache invalidé : memory_gpkg/population_grid_agglo_CTS.gpkg
    ✓ cache invalidé : memory_pbf/agglo_osm_pbf_CTS.osm.pbf
    ✓ cache invalidé : memory_ttm/ttm_CTS.parquet


Processing Files (1 / 1): 100%|██████████| 16.6MB / 16.6MB, 1.21MB/s  
New Data Upload: 100%|██████████| 16.6MB / 16.6MB, 1.21MB/s  


✓ Toulouse_GTFS.zip écrasé localement + poussé sur HF
Chargement du fichier GTFS : data/GTFS/Toulouse_GTFS.zip
✓ GTFS chargé avec succès
  Invalidation des caches dérivés pour 'Tisséo'...
    ✓ cache invalidé : memory_csv_agglo/decoupage_agglo_Tisséo.csv
    ✓ cache invalidé : memory_gpkg/population_grid_agglo_Tisséo.gpkg
    ✓ cache invalidé : memory_pbf/agglo_osm_pbf_Tisséo.osm.pbf
    ✓ cache invalidé : memory_ttm/ttm_Tisséo.parquet

10 GTFS mis à jour. Relance l'analyse (app ou scripts/run_benchmark_batch.py) pour recalculer leurs indicateurs.


In [5]:
# supprime les fichiers temporaires dans data/GTFS_temp/ 

In [6]:
# supprime l'ensemble des fichiers dans data/GTFS_temp — nettoyage une
# fois la comparaison/bascule ci-dessus terminée (~200+ Mo de GTFS
# téléchargés + gtfs_sources_temp.json + compare_GTFS.csv), pour ne pas
# laisser traîner ce dossier temporaire d'une exécution à l'autre.

fichiers_supprimes = 0
for nom in os.listdir(DOSSIER_TEMP):
    chemin = os.path.join(DOSSIER_TEMP, nom)
    if os.path.isfile(chemin):
        os.remove(chemin)
        fichiers_supprimes += 1

print(f"{fichiers_supprimes} fichier(s) supprimé(s) dans {DOSSIER_TEMP}")


57 fichier(s) supprimé(s) dans data/GTFS_temp
